# Official Recent Stats Knockout Projection

This notebook uses only the latest official FIFA team statistics and the real Round-of-32 bracket to project the remaining knockout rounds.

It deliberately ignores the historical ensemble so we can compare a pure recent-form view against the main model snapshots.

In [6]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'src').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    fallback = Path(r"C:\Users\PSA-Airbyte\autodev\projects\worldcup-prediction")
    if (fallback / 'src').exists():
        PROJECT_ROOT = fallback
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root containing 'src'.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.fifa_official import load_official_round_of_32
from src.features.official_team_stats import load_official_team_stats_features
from src.simulation.knockout_stage import FINAL_MATCH_NUMBER, QF_MATCH_NUMBERS, R16_MATCH_NUMBERS, SF_MATCH_NUMBERS, THIRD_PLACE_MATCH_NUMBER

In [7]:
recent_features = load_official_team_stats_features(require_resolved_state=False)
round_of_32 = load_official_round_of_32()

recent_features['recent_power'] = (
    0.24 * recent_features['official_recent_form_index']
    + 0.18 * recent_features['official_attack_signal']
    + 0.18 * recent_features['official_defense_signal']
    + 0.12 * recent_features['official_control_signal']
    + 0.10 * recent_features['official_goalkeeping_index']
    + 0.08 * recent_features['official_physical_index']
    + 0.10 * recent_features['official_xg_signal']
)
recent_lookup = recent_features.set_index('team').to_dict(orient='index')

display(recent_features.sort_values('recent_power', ascending=False)[[
    'team',
    'recent_power',
    'official_recent_form_index',
    'official_attack_signal',
    'official_defense_signal',
    'official_control_signal',
]].head(16))
display(round_of_32)

,team,recent_power,official_recent_form_index,official_attack_signal,official_defense_signal,official_control_signal
7,Canada,0.510840,0.4610,0.7245,0.3612,0.5305
40,Spain,0.434880,0.4666,0.5359,0.2685,0.9272
18,Germany,0.433824,0.4713,0.7139,0.2434,0.3911
45,United States,0.421666,0.4002,0.6031,0.1791,0.5145
4,Belgium,0.367254,0.3336,0.5388,0.1046,0.5072
16,England,0.361218,0.3606,0.5570,0.0537,0.5630
17,France,0.338626,0.4085,0.5588,0.1466,0.5064
27,Morocco,0.282704,0.3194,0.4188,0.1231,0.4997
42,Switzerland,0.269022,0.3082,0.4968,0.0784,0.4125
6,Brazil,0.267866,0.2641,0.4422,-0.0218,0.3703


,annex_c,match_number,date,home_team,away_team,home_path,away_path
0,M73,73,2026-06-28T19:00:00Z,South Africa,Canada,2A,2B
1,M74,74,2026-06-29T20:30:00Z,Germany,Paraguay,1E,3ABCDF
2,M75,75,2026-06-30T01:00:00Z,Netherlands,Morocco,1F,2C
3,M76,76,2026-06-29T17:00:00Z,Brazil,Japan,1C,2F
4,M77,77,2026-06-30T21:00:00Z,France,Sweden,1I,3CDFGH
5,M78,78,2026-06-30T17:00:00Z,Ivory Coast,Norway,2E,2I
6,M79,79,2026-07-01T01:00:00Z,Mexico,Ecuador,1A,3CEFHI
7,M80,80,2026-07-01T16:00:00Z,England,DR Congo,1L,3EHIJK
8,M81,81,2026-07-02T00:00:00Z,United States,Bosnia and Herzegovina,1D,3BEFIJ
9,M82,82,2026-07-01T20:00:00Z,Belgium,Senegal,1G,3AEHIJ


In [8]:
def recent_match_projection(home_team: str, away_team: str, match_id: str, annex_c: str) -> dict[str, object]:
    home = recent_lookup[home_team]
    away = recent_lookup[away_team]
    diff = float(home['recent_power'] - away['recent_power'])
    home_advance_probability = 1.0 / (1.0 + math.exp(-3.6 * diff))
    away_advance_probability = 1.0 - home_advance_probability
    winner = home_team if home_advance_probability >= away_advance_probability else away_team
    loser = away_team if winner == home_team else home_team
    return {
        'match_id': match_id,
        'annex_c': annex_c,
        'home_team': home_team,
        'away_team': away_team,
        'winner': winner,
        'loser': loser,
        'home_advance_probability': round(home_advance_probability, 4),
        'away_advance_probability': round(away_advance_probability, 4),
        'recent_power_diff': round(diff, 4),
    }


def next_round_pairings(results: list[dict[str, object]], match_ids: list[str], prefix: str) -> list[dict[str, str]]:
    pairings: list[dict[str, str]] = []
    for index in range(0, len(results), 2):
        left = results[index]
        right = results[index + 1]
        pairings.append(
            {
                'match_id': f'{prefix}-{index // 2 + 1}',
                'annex_c': match_ids[index // 2],
                'home_team': str(left['winner']),
                'away_team': str(right['winner']),
            }
        )
    return pairings

In [9]:
projected_round_of_32 = [
    recent_match_projection(row.home_team, row.away_team, f'R32-{index}', str(row.annex_c))
    for index, row in enumerate(round_of_32.itertuples(index=False), start=1)
]

round_of_16_pairings = next_round_pairings(projected_round_of_32, R16_MATCH_NUMBERS, 'R16')
projected_round_of_16 = [
    recent_match_projection(pairing['home_team'], pairing['away_team'], pairing['match_id'], pairing['annex_c'])
    for pairing in round_of_16_pairings
]

quarter_final_pairings = next_round_pairings(projected_round_of_16, QF_MATCH_NUMBERS, 'QF')
projected_quarter_finals = [
    recent_match_projection(pairing['home_team'], pairing['away_team'], pairing['match_id'], pairing['annex_c'])
    for pairing in quarter_final_pairings
]

semi_final_pairings = next_round_pairings(projected_quarter_finals, SF_MATCH_NUMBERS, 'SF')
projected_semi_finals = [
    recent_match_projection(pairing['home_team'], pairing['away_team'], pairing['match_id'], pairing['annex_c'])
    for pairing in semi_final_pairings
]

third_place_pairing = {
    'match_id': 'THIRD-1',
    'annex_c': THIRD_PLACE_MATCH_NUMBER,
    'home_team': projected_semi_finals[0]['loser'],
    'away_team': projected_semi_finals[1]['loser'],
}
projected_third_place = recent_match_projection(
    third_place_pairing['home_team'],
    third_place_pairing['away_team'],
    third_place_pairing['match_id'],
    third_place_pairing['annex_c'],
)

final_pairing = {
    'match_id': 'FINAL-1',
    'annex_c': FINAL_MATCH_NUMBER,
    'home_team': projected_semi_finals[0]['winner'],
    'away_team': projected_semi_finals[1]['winner'],
}
projected_final = recent_match_projection(
    final_pairing['home_team'],
    final_pairing['away_team'],
    final_pairing['match_id'],
    final_pairing['annex_c'],
)

display(pd.DataFrame(projected_round_of_32))
display(pd.DataFrame(projected_round_of_16))
display(pd.DataFrame(projected_quarter_finals))
display(pd.DataFrame(projected_semi_finals))

,match_id,annex_c,home_team,away_team,winner,loser,home_advance_probability,away_advance_probability,recent_power_diff
0,R32-1,M73,South Africa,Canada,Canada,South Africa,0.2102,0.7898,-0.3677
1,R32-2,M74,Germany,Paraguay,Germany,Paraguay,0.8890,0.1110,0.5781
2,R32-3,M75,Netherlands,Morocco,Morocco,Netherlands,0.3406,0.6594,-0.1835
3,R32-4,M76,Brazil,Japan,Brazil,Japan,0.7734,0.2266,0.3411
4,R32-5,M77,France,Sweden,France,Sweden,0.8582,0.1418,0.5002
5,R32-6,M78,Ivory Coast,Norway,Ivory Coast,Norway,0.5102,0.4898,0.0113
6,R32-7,M79,Mexico,Ecuador,Mexico,Ecuador,0.5885,0.4115,0.0994
7,R32-8,M80,England,DR Congo,England,DR Congo,0.8474,0.1526,0.4762
8,R32-9,M81,United States,Bosnia and Herzegovina,United States,Bosnia and Herzegovina,0.8935,0.1065,0.5909
9,R32-10,M82,Belgium,Senegal,Belgium,Senegal,0.6643,0.3357,0.1895


,match_id,annex_c,home_team,away_team,winner,loser,home_advance_probability,away_advance_probability,recent_power_diff
0,R16-1,M89,Canada,Germany,Canada,Germany,0.5689,0.4311,0.0770
1,R16-2,M90,Morocco,Brazil,Morocco,Brazil,0.5134,0.4866,0.0148
2,R16-3,M91,France,Ivory Coast,France,Ivory Coast,0.7096,0.2904,0.2481
3,R16-4,M92,Mexico,England,England,Mexico,0.3664,0.6336,-0.1522
4,R16-5,M93,United States,Belgium,United States,Belgium,0.5488,0.4512,0.0544
5,R16-6,M94,Portugal,Spain,Spain,Portugal,0.3012,0.6988,-0.2337
6,R16-7,M95,Switzerland,Argentina,Switzerland,Argentina,0.5737,0.4263,0.0825
7,R16-8,M96,Colombia,Egypt,Egypt,Colombia,0.3500,0.6500,-0.1720


,match_id,annex_c,home_team,away_team,winner,loser,home_advance_probability,away_advance_probability,recent_power_diff
0,QF-1,M97,Canada,Morocco,Canada,Morocco,0.6945,0.3055,0.2281
1,QF-2,M98,France,England,England,France,0.4797,0.5203,-0.0226
2,QF-3,M99,United States,Spain,Spain,United States,0.4881,0.5119,-0.0132
3,QF-4,M100,Switzerland,Egypt,Switzerland,Egypt,0.5217,0.4783,0.0241


,match_id,annex_c,home_team,away_team,winner,loser,home_advance_probability,away_advance_probability,recent_power_diff
0,SF-1,M101,Canada,England,Canada,England,0.6315,0.3685,0.1496
1,SF-2,M102,Spain,Switzerland,Spain,Switzerland,0.6450,0.3550,0.1659


In [10]:
summary = pd.DataFrame(
    [
        {'stage': 'Third place', **projected_third_place},
        {'stage': 'Final', **projected_final},
    ]
)
display(summary[['stage', 'annex_c', 'home_team', 'away_team', 'winner', 'home_advance_probability', 'away_advance_probability', 'recent_power_diff']])
print('Recent-stats-only champion:', projected_final['winner'])

,stage,annex_c,home_team,away_team,winner,home_advance_probability,away_advance_probability,recent_power_diff
0,Third place,M103,England,Switzerland,England,0.5822,0.4178,0.0922
1,Final,M104,Canada,Spain,Canada,0.5679,0.4321,0.0760


Recent-stats-only champion: Canada
